# 📖 Notebook 4: Production Patterns for ID Generation

The first three notebooks taught you *what* the formats look like. This one is about *living with them in production*:

- How do you actually assign **worker IDs** to a fleet of machines?
- How should you **store** UUIDs/ULIDs/Snowflakes in your database?
- How do you **decode** a snowflake from a real system (Twitter/X, Discord)?
- What's **NanoID** and where does it fit?
- Common gotchas you'll only hit in production.

## Learning Objectives

- Pick a worker-ID assignment strategy (env var, hash-of-hostname, k8s pod ordinal, coordinator)
- Know how to store IDs efficiently in Postgres / MySQL
- Decode an existing Discord snowflake back into a wall-clock timestamp
- Understand NanoID and when to choose it over UUID/ULID

## 🛠️ Setup

This notebook uses **only the Python standard library + `pydantic`**.

```bash
cd 01-foundations/id-generation
uv sync
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

## 1️⃣ Assigning worker IDs in real fleets

A Snowflake generator is only correct if **every running process has a unique `worker_id`**.
Two processes sharing an ID will eventually emit duplicate IDs in the same millisecond.

Here are the four most common strategies, ordered from simplest to most robust:

| Strategy | How it works | Pros | Cons |
|---|---|---|---|
| **Env var / config** | Operator hard-codes `WORKER_ID=7` per deployment | Trivial | Manual, error-prone |
| **Hash of hostname/IP** | Take last 10 bits of `hash(hostname)` | Zero coordination | Collisions possible |
| **K8s StatefulSet ordinal** | Pod name `app-3` → worker_id 3 | Automatic, stable | Only works for stateful sets |
| **Zookeeper / etcd lease** | Workers grab a free slot from a coordinator | Robust at scale | Needs another system |

Let's implement the first three — the fourth needs a real coordinator.

In [ ]:
import os, socket, hashlib

MAX_WORKER = 1023  # 10 bits

def worker_from_env(default=None):
    """Strategy 1: trust the operator. Read WORKER_ID from the environment."""
    raw = os.environ.get("WORKER_ID")
    if raw is None:
        if default is None:
            raise RuntimeError("WORKER_ID env var not set")
        return default
    wid = int(raw)
    if not 0 <= wid <= MAX_WORKER:
        raise ValueError(f"WORKER_ID={wid} outside 0..{MAX_WORKER}")
    return wid

def worker_from_hostname():
    """Strategy 2: derive a worker id from the hostname/IP.

    Good for stateless services where you do not want to manage IDs.
    Use a strong hash so different hostnames spread evenly."""
    host = socket.gethostname()
    digest = hashlib.sha256(host.encode()).digest()
    # Take 2 bytes (16 bits) and mask down to 10 bits
    return int.from_bytes(digest[:2], "big") & MAX_WORKER

def worker_from_pod_ordinal():
    """Strategy 3: Kubernetes StatefulSet pods are named `app-0`, `app-1`, ...

    The ordinal at the end is a perfect worker id - guaranteed unique
    by Kubernetes itself."""
    pod = os.environ.get("HOSTNAME", "app-0")  # k8s sets HOSTNAME = pod name
    ordinal = pod.rsplit("-", 1)[-1]
    if not ordinal.isdigit():
        raise ValueError(f"Pod name {pod!r} does not end in a number")
    wid = int(ordinal)
    if not 0 <= wid <= MAX_WORKER:
        raise ValueError(f"Ordinal {wid} outside 0..{MAX_WORKER}")
    return wid

# Demo: try all three
print("From env:      ", worker_from_env(default=42))
print("From hostname: ", worker_from_hostname(), f"(host={socket.gethostname()!r})")
print("From pod:      ", worker_from_pod_ordinal())


### Why this matters

Imagine you autoscale 50 pods. With **env var**, somebody has to assign 50 unique IDs — tedious and bug-prone.

With **hashing**, the chance of *any* two of N machines colliding (birthday paradox) is roughly `N² / 2¹¹` for 10 bits. With N=20 you have ~20% chance of collision. **Bad.**

With **k8s ordinals** or a **coordinator**, collisions are impossible by construction. That's why production systems usually pick one of those two.

## 2️⃣ Storing IDs efficiently in your database

A 16-byte UUID stored as a **string** (`"550e8400-e29b-41d4-a716-446655440000"`) takes **36 bytes** in your row, plus extra in the index.

The same UUID stored as a **native binary type** (`uuid` in Postgres, `BINARY(16)` in MySQL) takes only **16 bytes** — less than half.

For a billion-row table that difference is **~20 GB** on disk and a much smaller index that fits in RAM.

| Format | Bytes (binary) | Postgres column | MySQL column |
|---|---|---|---|
| UUID v4 / v7 | 16 | `uuid` | `BINARY(16)` (or `UUID` 8.0+) |
| ULID | 16 | `uuid` (same 128 bits) | `BINARY(16)` |
| KSUID | 20 | `bytea` or `bit(160)` | `BINARY(20)` |
| Snowflake | 8 | `bigint` | `BIGINT UNSIGNED` |

Let's prove the size point with a quick measurement.

In [ ]:
import sys, uuid

u = uuid.uuid4()
as_string = str(u)              # what most apps store by default
as_bytes  = u.bytes             # what Postgres `uuid` type stores

print(f"As string ({len(as_string)} chars):  {as_string}")
print(f"As bytes  ({len(as_bytes)} bytes):   {as_bytes.hex()}")
print()
print("Storage in a real DB row (no Python object overhead):")
print(f"  TEXT/VARCHAR(36):   36 bytes per row")
print(f"  Postgres uuid type: 16 bytes per row  ({(36-16)/36:.0%} smaller)")
print()
print("Per billion rows that is ~20 GB saved on disk + a much smaller index.")


### Postgres-specific tips

- Use the native `uuid` column type, not `text`. Indexes are smaller and comparisons are faster.
- For UUIDv7/ULID stored in a `uuid` column, the **byte order is preserved**, so an index on the column is naturally sorted by time. Inserts hit the end of the B-tree — great write performance.
- For ULID specifically, you can store the same 16 raw bytes in a `uuid` column and convert to/from the Base32 string in your application layer.

## 3️⃣ Decoding a real-world snowflake

Twitter (now X) and Discord both use Snowflake-style IDs. Discord publishes its bit layout in the [public docs](https://discord.com/developers/docs/reference#snowflakes):

```
| 42 bits timestamp (ms since Discord epoch) | 5 bits worker | 5 bits process | 12 bits sequence |
```

Discord's epoch is **1420070400000** (2015-01-01T00:00:00Z).

Let's decode a real Discord user ID. The classic example is the bot ID `175928847299117063`.

In [ ]:
from datetime import datetime, timezone

DISCORD_EPOCH_MS = 1420070400000  # 2015-01-01

def decode_discord(snowflake):
    sequence       =  snowflake        & 0xFFF        # 12 bits
    process_id     = (snowflake >> 12) & 0x1F         # 5 bits
    worker_id      = (snowflake >> 17) & 0x1F         # 5 bits
    ts_offset_ms   = (snowflake >> 22)                # 42 bits
    unix_ms = ts_offset_ms + DISCORD_EPOCH_MS
    return {
        "raw": snowflake,
        "created_at_utc": datetime.fromtimestamp(unix_ms/1000, tz=timezone.utc).isoformat(),
        "worker_id": worker_id,
        "process_id": process_id,
        "sequence": sequence,
    }

# Example: a known Discord snowflake
example = 175928847299117063
for k, v in decode_discord(example).items():
    print(f"  {k:>15s}: {v}")


☝️ Notice we extracted a real wall-clock time from a 64-bit integer with no API call. **This is a superpower of time-ordered IDs**: you can answer *"when was this row created?"* without storing a separate `created_at` column.

> Pro tip: if you've ever wondered why some companies leak product launch dates by accident — sometimes it's because they used a snowflake-style ID and a bored engineer decoded the smallest one.

## 4️⃣ NanoID — the short, URL-friendly cousin

Sometimes you don't need 128 bits of entropy or millisecond ordering. You just want a **short, opaque, URL-safe** string for things like:

- Shareable links (think `youtu.be/dQw4w9WgXcQ`)
- Coupon codes
- Public-facing entity slugs

That's where [NanoID](https://github.com/ai/nanoid) shines:

- **Shorter than UUID** (default 21 chars vs 36)
- **URL-safe alphabet** (`A-Za-z0-9_-`)
- **Customisable length and alphabet** — pick your own collision/length trade-off
- **Not time-ordered** — it's pure randomness, like UUIDv4

A 21-char NanoID has ≈126 bits of entropy — practically the same as a UUIDv4 — but in **42% fewer characters**.

Here's a from-scratch implementation:

In [ ]:
import secrets, math

ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789_-"  # 64 chars
assert len(ALPHABET) == 64  # power of 2 -> unbiased sampling with bit masks

def nanoid(length=21):
    """Generate a NanoID. Each char carries log2(64) = 6 bits of entropy."""
    # secrets.token_bytes is cryptographically secure
    raw = secrets.token_bytes(length)
    # Map each byte to one alphabet character (mask to 6 bits = 0..63)
    return "".join(ALPHABET[b & 0x3F] for b in raw)

for _ in range(5):
    print(nanoid())

print()
print(f"Default length: 21 chars  ->  {21*math.log2(64):.0f} bits of entropy")
print(f"UUIDv4 (36 chars):           122 bits of entropy")
print(f"NanoID is shorter AND nearly as collision-resistant.")


### Length / collision trade-off cheat sheet

You can shorten NanoIDs at the cost of more collisions per N IDs generated.
With a 64-char alphabet:

| Length | Bits | IDs before 1% collision chance |
|---|---|---|
| 8  | 48  | ~2.4 million |
| 12 | 72  | ~10 billion |
| 16 | 96  | ~10 trillion |
| 21 | 126 | ~10¹⁸ (essentially never) |

Pick the smallest size that fits your traffic. For shareable links seen by millions, **12 characters is plenty**.

## 5️⃣ Production gotchas you'll only learn the hard way

A short but expensive list:

1. **Don't expose snowflake IDs in public URLs without thinking.**
   They leak creation order *and* timestamp. Competitors can estimate your daily signup count by sampling user IDs at 24-hour intervals — a real OSINT trick.

2. **Pick the epoch carefully.** A 41-bit ms timestamp gives you 69 years. If your epoch is 2010, you wrap in 2079 — probably fine. If your epoch is 1970 (Unix epoch), you wrap in **2039**, sooner than expected.

3. **Don't change the worker ID of a running process.** Two IDs minted at the same ms with different worker IDs are different — but nobody enforces that the *new* ID is greater than the previous one. Restart the process between worker reassignments.

4. **NTP step adjustments still happen.** Even with `time.monotonic()` defences, a VM resume after a long pause can leap forward by minutes. Most Snowflake libraries treat *forward* leaps as fine but *backward* leaps as fatal.

5. **Test what happens when you exhaust the sequence.** A burst of >4096 IDs in 1 ms is normal during traffic spikes. A naive generator will block; a safer one will reject with a clear error so callers can retry instead of timing out.

6. **For UUIDs in URLs, prefer Base32 or Base64url over hex.** It's shorter (22 chars Base64url vs 36 chars hex) and still URL-safe.

Let's demonstrate the last one — encoding a UUID as 22 chars instead of 36:

In [ ]:
import uuid, base64

u = uuid.uuid4()
hex_form    = str(u)                                          # 36 chars
b64url_form = base64.urlsafe_b64encode(u.bytes).rstrip(b"=").decode()  # 22 chars

print(f"UUID hex     ({len(hex_form):>2} chars):  {hex_form}")
print(f"UUID b64url  ({len(b64url_form):>2} chars):  {b64url_form}")
print(f"\nBoth carry the same 128 bits. b64url is {(36-22)/36:.0%} shorter -- great for URLs.")


## 🎯 Recap

You now have the practical knowledge to use these IDs in real systems:

- **Worker IDs**: prefer k8s ordinals or a coordinator over manual env vars at scale
- **Storage**: native binary types (`uuid`, `BIGINT`) save ~50% space and speed up indexes
- **Decoding**: time-ordered IDs double as built-in `created_at` columns — for free
- **NanoID**: the right choice for short, opaque, URL-friendly IDs
- **Gotchas**: epoch wrap-around, NTP jumps, sequence exhaustion, info leaks

### Where to go next

- Read the actual specs:
  - [RFC 9562](https://datatracker.ietf.org/doc/rfc9562/) — UUIDv6/v7/v8
  - [ULID spec](https://github.com/ulid/spec)
  - [KSUID announcement](https://segment.com/blog/a-brief-history-of-the-uuid/) by Segment
  - [Discord docs](https://discord.com/developers/docs/reference#snowflakes) — public snowflake layout
- Try the well-maintained Python libraries when you'd rather not roll your own:
  - `uuid6` (UUIDv7) — `pip install uuid6`
  - `python-ulid`
  - `svix-ksuid`
  - `nanoid`
- Skim your database's docs on UUID indexing — Postgres, MySQL, and SQL Server all behave a little differently.